In [1]:
#Loading the Excel File and libraries
import pandas as pd
import numpy as np

file_path = '/content/Global Oil and Gas Extraction Tracker - 2024-02-22_1039.xlsx'
xl = pd.ExcelFile(file_path)

In [2]:
#setting dataframes with main, P&R
main_data_df = xl.parse('Main data')
production_reserves_df = xl.parse('Production & reserves')


In [3]:
#Filtering for 2022 and 2023 data
main_data_df["FID Year"] = main_data_df["FID Year"].astype(str).str.replace(" \(expected\)", "", regex=True)
filtered_main_data_df = main_data_df[main_data_df["FID Year"].isin(["2022", "2023"])]


In [4]:
# Convert 'Data year' to integers in the 'production_reserves_df'
production_reserves_df['Data year'] = pd.to_numeric(production_reserves_df['Data year'], errors='coerce').fillna(-1).astype(int)


In [5]:
#Define functions for getting the most recent value, calculate the total reserves, calculate the total production and define the conversion factor
# Function to get the most recent value based on the criteria
def get_most_recent_value(unit_id, prod_res, units, df):
    # Filter based on Unit ID, Production/reserves, and Units (converted)
    filtered = df[
        (df["Unit ID"] == unit_id) &
        (df["Production/reserves"] == prod_res) &
        (df["Units (converted)"] == units)
    ]

    # Sort by Data year and get the most recent entry
    filtered = filtered.sort_values(by="Data year", ascending=False)
    return filtered.iloc[0]["Quantity (converted)"] if not filtered.empty else np.nan

# Function to calculate the total reserves
def calculate_total_reserves(row):
    if pd.notna(row['Reserves- Hydrocarbons (unspecified)']):
        return row['Reserves- Hydrocarbons (unspecified)']
    else:
        # Convert gas reserves to boe
        gas_in_boe = row['Reserves- Gas'] * conversion_factor if pd.notna(row['Reserves- Gas']) else 0
        oil_reserves = row['Reserves - Oil'] if pd.notna(row['Reserves - Oil']) else 0
        return gas_in_boe + oil_reserves

# Function to calculate the total production
def calculate_total_production(row):
    if pd.notna(row['Production - Hydrocarbons (unspecified)']):
        return row['Production - Hydrocarbons (unspecified)']
    else:
        # Convert gas production to boe
        gas_in_boe = row['Production - Gas'] * conversion_factor if pd.notna(row['Production - Gas']) else 0
        oil_production = row['Production - Oil'] if pd.notna(row['Production - Oil']) else 0
        return gas_in_boe + oil_production

# Conversion factor from million m³ to million boe for gas
conversion_factor = 5.883 / 1000


In [6]:
# Populate reserves and production columns with the most recent data
filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
filtered_main_data_df["Reserves- Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million boe", production_reserves_df))

filtered_main_data_df["Production - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million bbl/y", production_reserves_df))
filtered_main_data_df["Production - Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million m³/y", production_reserves_df))
filtered_main_data_df["Production - Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million boe/y", production_reserves_df))

# Calculate total reserves and production
filtered_main_data_df['Reserves- Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_reserves, axis=1)
filtered_main_data_df['Production - Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_production, axis=1)


<ipython-input-6-e0a58fe69be6>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
<ipython-input-6-e0a58fe69be6>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
<ipython-input-6-e

In [7]:
filtered_main_data_df

,Unit ID,Unit Name,Unit name local script,Fuel type,Unit type,Country,"Subnational unit (province, state)",Latitude,Longitude,Location accuracy,...,Government unit ID,Wiki URL,Reserves - Oil,Reserves- Gas,Reserves- Hydrocarbons (unspecified),Production - Oil,Production - Gas,Production - Hydrocarbons (unspecified),"Reserves- Total (Oil, Gas and Hydrocarbons)","Production - Total (Oil, Gas and Hydrocarbons)"
315,OG0001255,N05-A,NaN,oil and gas,field,Netherlands,NaN,53.678000,6.362000,exact,...,2483538445,https://www.gem.wiki/N05-A_Oil_and_Gas_Field_%...,NaN,8.500000e+03,NaN,NaN,NaN,NaN,50.005500,0
521,OG0001800,Neptun Deep,NaN,gas,project,Romania,NaN,43.730000,30.904800,exact,...,NaN,https://www.gem.wiki/Neptun_Deep_Gas_Project_%...,NaN,1.000000e+05,NaN,NaN,NaN,NaN,588.300000,0
697,OG0002076,Jackdaw,NaN,oil and gas,field,United Kingdom,NaN,56.901036,2.380758,exact,...,NaN,https://www.gem.wiki/Jackdaw_Oil_and_Gas_Field...,NaN,NaN,185.0,NaN,NaN,NaN,185.000000,0
853,OG0002274,Lakach,Lakach,oil and gas,field,Mexico,Federal Offshore,19.051679,-95.265979,exact,...,NaN,https://www.gem.wiki/Lakach_Oil_and_Gas_Field_...,0.000000,2.400963e+04,NaN,NaN,NaN,NaN,141.248639,0
945,OG0002406,Trión,Trión,oil and gas,field,Mexico,Federal Offshore,25.764290,-95.385284,exact,...,NaN,https://www.gem.wiki/Tri%C3%B3n_Oil_and_Gas_Fi...,471.000000,1.203455e+04,NaN,NaN,NaN,NaN,541.799235,0
1014,OG0002677,Pit,Pit,oil and gas,field,Mexico,Federal Offshore,19.769700,-92.268700,exact,...,AE-0165-M-Campeche Oriente,https://www.gem.wiki/Pit_Oil_and_Gas_Field_%28...,459.230000,1.603285e+03,NaN,NaN,NaN,NaN,468.662124,0
2433,OG0010918,Cypre,Cypre,gas,field,Trinidad and Tobago,NaN,10.030985,-60.289759,approximate,...,NaN,https://www.gem.wiki/Cypre_Gas_Field_%28Trinid...,NaN,2.183208e+04,NaN,NaN,NaN,NaN,128.438142,0
2891,OG0011918,Kasawari,NaN,oil and gas,field,Malaysia,NaN,4.980300,112.978700,exact,...,NaN,https://www.gem.wiki/Kasawari_Oil_and_Gas_Fiel...,NaN,8.412176e+04,NaN,NaN,NaN,NaN,494.888322,0
3018,OG0012079,Marjoram,NaN,oil and gas,field,Malaysia,NaN,NaN,NaN,NaN,...,NaN,https://www.gem.wiki/Marjoram_Oil_and_Gas_Fiel...,NaN,NaN,68.0,NaN,NaN,NaN,68.000000,0
3023,OG0012085,Rosmari,NaN,oil and gas,field,Malaysia,NaN,NaN,NaN,NaN,...,NaN,https://www.gem.wiki/Rosmari_Oil_and_Gas_Field...,NaN,NaN,68.0,NaN,NaN,NaN,68.000000,0


In [8]:
output_excel_path = '/content/FID20222023 20240222.xlsx'
filtered_main_data_df.to_excel(output_excel_path, index=False)